# Stage D (Part 3) — Multi-Horizon Targets

**Purpose.** From already-engineered daily tables, build horizon-specific targets for forecasting (e.g., +1, +2, +3, +5, +7 days).  
This notebook **does not** redo feature engineering or do any splits; it just appends `y_h{H}` and saves per-horizon flat datasets.

---

## Inputs
- `data/modeling/datasets/train_basin_daily_exogenous.parquet`  
  (engineered exogenous features at daily resolution, includes `discharge_cms`)
- `data/modeling/datasets/train_basin_daily_arx.parquet`  
  (ARX version with discharge lags/rolling stats)
- They should have been created from Stage D (Part 2)

## Outputs (per horizon `H`)
- `data/modeling/datasets/train_basin_daily_exogenous_h{H}.parquet`
- `data/modeling/datasets/train_basin_daily_arx_h{H}.parquet`

Each output is the same as the base table **plus** a single target column:
- `y_h{H}` = future discharge for that basin at **t+H** days.

---

## What this notebook does
1. **Load & sort** the base parquet(s) by `basin_id, date_local`.
2. For each `H` in `HORIZONS = [1, 2, 3, 5, 7]`:
   - Create `y_h{H}` via `groupby('basin_id')['discharge_cms'].shift(-H)`.
   - **Drop** rows where `y_h{H}` is missing (tail rows per basin).
   - **Save** the per-horizon parquet for both **exogenous** and **ARX** variants.
3. Print a short summary of rows kept/dropped for each horizon.

---

## Notes & Assumptions
- All QC (e.g., `qc_any`) and feature engineering were done **upstream**.
- This step **does not** create splits; splitting is deferred to the next notebook.
- `groupby(...).shift(-H)` prevents any cross-basin leakage.
- Features are unchanged; only a new target column is added and tail rows are removed.
- Keep the same column names and file layout for easy downstream use.

---

## Quick sanity checks (optional)
- For a chosen basin/date `t`, verify `y_h3(t)` equals that basin’s `discharge_cms` at `t+3`.
- Row counts drop by ≈`H` per basin (extra drops if the future target is missing).
- No NaNs in `y_h{H}`; feature columns match the base tables.


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# === Resolve Project Root ===

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/qingfangliu/bhutan_climate_modeling


In [4]:
# --- Config ---
HORIZONS = [1, 2, 3, 5, 7]

DATASETS_DIR = PROJECT_ROOT / "data" / "modeling" / "datasets"
exog_path = DATASETS_DIR / "train_basin_daily_exogenous.parquet"
arx_path  = DATASETS_DIR / "train_basin_daily_arx.parquet"

In [5]:
# --- Load ---
train_exog = pd.read_parquet(exog_path)
te = pd.read_parquet(arx_path)  # ARX version

# --- Ensure sort (per-basin chronological) ---
if "date_local" in train_exog.columns:
    train_exog["date_local"] = pd.to_datetime(train_exog["date_local"])
if "date_local" in te.columns:
    te["date_local"] = pd.to_datetime(te["date_local"])

train_exog = (train_exog
              .sort_values(["basin_id", "date_local"])
              .reset_index(drop=True))
te = (te
      .sort_values(["basin_id", "date_local"])
      .reset_index(drop=True))

In [6]:
train_exog.head()

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,Plinthosols_pct,Podzols_pct,Regosols_pct,Solonchaks_pct,Solonetz_pct,Stagnosols_pct,Umbrisols_pct,Vertisols_pct,discharge_cms,qc_any
0,3.0,2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.783,False
1,3.0,2000-01-02,-0.000222,NaN,NaN,NaN,NaN,0.000018,NaN,NaN,...,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.620,False
2,3.0,2000-01-03,-0.000248,NaN,NaN,NaN,NaN,0.000029,NaN,NaN,...,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.565,False
3,3.0,2000-01-04,-0.000225,-0.000695,NaN,NaN,NaN,0.000090,0.000136,NaN,...,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.031,False
4,3.0,2000-01-05,-0.000241,-0.000714,NaN,NaN,NaN,0.000015,0.000134,NaN,...,0.0,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.032,False


In [7]:
te.head()

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,discharge_cms,qc_any,q_lag_1d,q_lag_2d,q_lag_3d,q_lag_7d,q_lag_14d,q_lag_30d,q_roll7_mean,q_roll14_std
0,3.0,2000-01-31,-0.000218,-0.000624,-0.001396,-0.002778,-0.006060,0.000112,0.001292,0.002773,...,11.928,False,11.590,12.506,13.302,13.099,13.715,14.783,12.885714,0.568484
1,3.0,2000-02-01,-0.000181,-0.000598,-0.001385,-0.002764,-0.006019,0.000428,0.001114,0.002549,...,12.550,False,11.928,11.590,12.506,13.302,13.715,14.620,12.718429,0.627722
2,3.0,2000-02-02,-0.000175,-0.000574,-0.001364,-0.002677,-0.005945,0.000430,0.000970,0.002714,...,12.160,False,12.550,11.928,11.590,13.099,13.872,14.565,12.611000,0.604644
3,3.0,2000-02-03,-0.000127,-0.000483,-0.001296,-0.002598,-0.005847,0.002213,0.003071,0.004777,...,12.452,False,12.160,12.550,11.928,13.302,13.406,14.031,12.476857,0.572209
4,3.0,2000-02-04,-0.000202,-0.000503,-0.001308,-0.002636,-0.005809,0.000196,0.002838,0.004559,...,12.305,False,12.452,12.160,12.550,13.302,12.948,14.032,12.355429,0.552219


In [3]:
# --- Build & save per-horizon tables ---
for H in HORIZONS:
    # Exogenous
    exog_h = train_exog.copy()
    exog_h[f"y_h{H}"] = exog_h.groupby("basin_id")["discharge_cms"].shift(-H) # shift to create future target
    n_before = len(exog_h)
    exog_h = exog_h.dropna(subset=[f"y_h{H}"]).reset_index(drop=True) # drop rows where y_h{H} is NaN
    out_exog = DATASETS_DIR / f"train_basin_daily_exogenous_h{H}.parquet"
    exog_h.to_parquet(out_exog, index=False)
    print(f"[EXOG h={H}] {len(exog_h)} rows (dropped {n_before - len(exog_h)}) → {out_exog}")

    # ARX
    arx_h = te.copy()
    arx_h[f"y_h{H}"] = arx_h.groupby("basin_id")["discharge_cms"].shift(-H)
    n_before_arx = len(arx_h)
    arx_h = arx_h.dropna(subset=[f"y_h{H}"]).reset_index(drop=True)
    out_arx = DATASETS_DIR / f"train_basin_daily_arx_h{H}.parquet"
    arx_h.to_parquet(out_arx, index=False)
    print(f"[ARX  h={H}] {len(arx_h)} rows (dropped {n_before_arx - len(arx_h)}) → {out_arx}")

[EXOG h=1] 29140 rows (dropped 45) → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous_h1.parquet
[ARX  h=1] 29008 rows (dropped 3) → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_arx_h1.parquet
[EXOG h=2] 29137 rows (dropped 48) → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous_h2.parquet
[ARX  h=2] 29005 rows (dropped 6) → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_arx_h2.parquet
[EXOG h=3] 29134 rows (dropped 51) → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous_h3.parquet
[ARX  h=3] 29002 rows (dropped 9) → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_arx_h3.parquet
[EXOG h=5] 29128 rows (dropped 57) → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous_h5.parquet
[ARX  h=5] 28996 rows (drop

In [8]:
exog_h.head()

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,Podzols_pct,Regosols_pct,Solonchaks_pct,Solonetz_pct,Stagnosols_pct,Umbrisols_pct,Vertisols_pct,discharge_cms,qc_any,y_h7
0,3.0,2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.783,False,14.565
1,3.0,2000-01-02,-0.000222,NaN,NaN,NaN,NaN,0.000018,NaN,NaN,...,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.620,False,14.350
2,3.0,2000-01-03,-0.000248,NaN,NaN,NaN,NaN,0.000029,NaN,NaN,...,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.565,False,14.565
3,3.0,2000-01-04,-0.000225,-0.000695,NaN,NaN,NaN,0.000090,0.000136,NaN,...,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.031,False,14.189
4,3.0,2000-01-05,-0.000241,-0.000714,NaN,NaN,NaN,0.000015,0.000134,NaN,...,0.000046,0.0,0.0,0.0,0.0,0.0,0.0,14.032,False,14.136


In [9]:
arx_h.head()

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,qc_any,q_lag_1d,q_lag_2d,q_lag_3d,q_lag_7d,q_lag_14d,q_lag_30d,q_roll7_mean,q_roll14_std,y_h7
0,3.0,2000-01-31,-0.000218,-0.000624,-0.001396,-0.002778,-0.006060,0.000112,0.001292,0.002773,...,False,11.590,12.506,13.302,13.099,13.715,14.783,12.885714,0.568484,11.638
1,3.0,2000-02-01,-0.000181,-0.000598,-0.001385,-0.002764,-0.006019,0.000428,0.001114,0.002549,...,False,11.928,11.590,12.506,13.302,13.715,14.620,12.718429,0.627722,12.257
2,3.0,2000-02-02,-0.000175,-0.000574,-0.001364,-0.002677,-0.005945,0.000430,0.000970,0.002714,...,False,12.550,11.928,11.590,13.099,13.872,14.565,12.611000,0.604644,12.160
3,3.0,2000-02-03,-0.000127,-0.000483,-0.001296,-0.002598,-0.005847,0.002213,0.003071,0.004777,...,False,12.160,12.550,11.928,13.302,13.406,14.031,12.476857,0.572209,11.778
4,3.0,2000-02-04,-0.000202,-0.000503,-0.001308,-0.002636,-0.005809,0.000196,0.002838,0.004559,...,False,12.452,12.160,12.550,13.302,12.948,14.032,12.355429,0.552219,11.873


## Verify the saved datasets

In [11]:
# --- settings (change H to 1/2/5/7 to check others) ---
H = 3
DATASETS_DIR = PROJECT_ROOT / "data" / "modeling" / "datasets"
exog_base_path = DATASETS_DIR / "train_basin_daily_exogenous.parquet"
exog_h_path    = DATASETS_DIR / f"train_basin_daily_exogenous_h{H}.parquet"

# --- load & sort ---
base = pd.read_parquet(exog_base_path)
exog_h = pd.read_parquet(exog_h_path)
base["date_local"] = pd.to_datetime(base["date_local"])
exog_h["date_local"] = pd.to_datetime(exog_h["date_local"])
base = base.sort_values(["basin_id","date_local"]).reset_index(drop=True)
exog_h = exog_h.sort_values(["basin_id","date_local"]).reset_index(drop=True)

# 1) row-count sanity: about H rows dropped per basin (tail)
n0 = base.groupby("basin_id").size()
nH = exog_h.groupby("basin_id").size()
diff = (n0 - nH).rename("dropped_tail_rows").reset_index()
print(diff.head())  # preview a few basins

# 2) target column should have no NaNs
print(f"NaNs in y_h{H}: ", exog_h[f"y_h{H}"].isna().sum())

# 3) value alignment check in one basin at a middle date
bid = exog_h["basin_id"].iloc[0]
df0 = base[base["basin_id"] == bid].sort_values("date_local").reset_index(drop=True)

i = 10  # pick a middle row to avoid edges
if i + H >= len(df0):  # just in case the basin is tiny
    i = 0

date_t = df0.loc[i, "date_local"]
future_q = df0.loc[i + H, "discharge_cms"]

yh = exog_h.loc[
    (exog_h["basin_id"] == bid) & (exog_h["date_local"] == date_t),
    f"y_h{H}"
].iloc[0]

print(f"[basin {bid}] on {date_t.date()}: y_h{H} = {yh}  |  discharge_cms(t+{H}) = {future_q}")


   basin_id  dropped_tail_rows
0       3.0                  3
1       6.0                  3
2       8.0                 45
NaNs in y_h3:  0
[basin 3.0] on 2000-01-11: y_h3 = 14.13599967956543  |  discharge_cms(t+3) = 14.13599967956543
